# 08 - Evaluate merges, collapsed seeds

This notebook evaluates every fresh merge using the same 2,000-log / 500-QA test subsets. QA evaluation uses the configured LLM judge.

In [ ]:
import exp_common as E
import pandas as pd

RUN_SEEDS = [42, 123, 777]
EVAL_LOGS_N = 2000
EVAL_QA_N = 500
BATCH_LOGS = 16
BATCH_QA = 4
JUDGE_N = 100

logs_eval = E.read_jsonl(E.PREPARED_DIR / "logs_test_eval.jsonl")[:EVAL_LOGS_N]
qa_eval = E.read_jsonl(E.PREPARED_DIR / "qa_test_eval.jsonl")[:EVAL_QA_N]
out_dir = E.RESULTS_DIR / "merged"
out_dir.mkdir(parents=True, exist_ok=True)

def eval_adapter(label, adapter_dir, seed, base_adapter_dir=None):
    lm = E.evaluate_logs(adapter_dir, logs_eval, out_dir / f"{label}_seed_{seed}_logs.json", batch_size=BATCH_LOGS, base_adapter_dir=base_adapter_dir)
    qm = E.evaluate_qa(adapter_dir, qa_eval, out_dir / f"{label}_seed_{seed}_qa.json", batch_size=BATCH_QA, judge_sample=JUDGE_N, base_adapter_dir=base_adapter_dir)
    return {
        "model": label, "seed": seed,
        **{f"logs_{k}": v for k,v in lm.items() if not isinstance(v,(list,dict))},
        **{f"qa_{k}": v for k,v in qm.items() if not isinstance(v,(list,dict))},
    }

rows = []
for seed in RUN_SEEDS:
    for scenario, anchor in [("base_logs_plus_qa", None), ("cpt_anchor_logs_plus_qa", E.ADAPTERS_DIR/"cpt"/f"seed_{seed}"/"adapter")]:
        for space in ["factor", "delta_w"]:
            for method in ["task_arithmetic", "ties", "model_soup", "dare_plus_task_arithmetic", "dare_plus_ties", "dare_plus_model_soup"]:
                adapter = E.MERGED_DIR / scenario / space / method / f"seed_{seed}"
                if (adapter / "adapter_model.safetensors").exists():
                    rows.append(eval_adapter(f"{scenario}__{space}__{method}", adapter, seed, anchor))
                else:
                    print("Missing:", adapter)

df = pd.DataFrame(rows)
df.to_csv(out_dir / "merged_metrics_raw.csv", index=False)
num_cols = [c for c in df.columns if c not in ["model","seed"] and pd.api.types.is_numeric_dtype(df[c])]
df.groupby("model")[num_cols].agg(["mean","std"]).to_csv(out_dir / "merged_metrics_mean_std.csv")
df

## DARE+TIES collapse diagnostic

This cell produces a compact diagnostic with recall, precision, F1, predicted anomaly rate and off-format rate.

In [ ]:
cols = ["model","seed","logs_recall_anomaly","logs_precision_anomaly","logs_f1_anomaly","logs_pred_anomaly_rate","logs_off_format_rate","logs_coverage_strict"]
collapse = df[[c for c in cols if c in df.columns]].copy()
collapse = collapse[collapse["model"].str.contains("dare_plus_ties", na=False)]
collapse.to_csv(out_dir / "dare_plus_ties_seed_diagnostic.csv", index=False)
collapse